# Multi-Armed Bandits

<center><img src="./img/bandit.jpg"/></center>

In this lab, we study the exploration-exploitation dilemma in the multi-armed bandit problem. Each arm provides a random reward with an unknown mean. The agent must learn which arms are promising while collecting rewards. Applications include online advertising and recommendation systems.

Unlike a general reinforcement learning problem, this setting has no state transitions: choosing an arm only reveals its reward. We assume stationary, independent reward distributions throughout the lab.

You will implement random, greedy, $\varepsilon$-greedy, UCB1 and Thompson sampling strategies, compare their average rewards and optimal-arm selection rates, and examine the effect of exploration parameters.

**Your task:** complete the fragments marked `<YOUR CODE GOES HERE>`, run all cells from top to bottom, and answer the questions at the end. The solution and task notebooks have the same text, experiments and checks; only the implementation fragments differ.

Small checks from `review_checks.py` follow each implementation stage, as in the A* lab. All assertions must pass before you interpret the experiment plots.

## Setup and reproducibility

Install the dependencies from `requirements.txt` in your notebook environment, then select its Python kernel. Run this notebook with `lab07-bandits` as the working directory so that `review_checks.py` and the image can be found.

Each `conduct_experiment` call resets NumPy's random seed (default `SEED = 42`). Re-running an experiment with the same parameters and seed reproduces the same histories in the same software environment, even after running other cells. Separate instances within an experiment consume different random draws. Pass another integer as `seed` to study variation across independent experiments. Direct calls to `run_bandits` can also specify `seed`.

The checks preserve the random state. Plots show cumulative average reward and the cumulative proportion of optimal-arm choices, averaged over independent instances in the comparison plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as st
from dataclasses import dataclass, field


sns.set_style("whitegrid")
sns.set_palette("colorblind")
SEED = 42

## Reward models and interaction history

We use two stationary reward models:

- **Bernoulli bandit:** reward 1 with probability $p$, otherwise 0; its mean is $p \in [0, 1]$.
- **Gaussian bandit:** reward $R \sim \mathcal{N}(p, \sigma^2)$, where $p$ is an unrestricted mean. All Gaussian experiments below use variance $\sigma^2 = 1$. NumPy's `normal` takes the standard deviation $\sigma$, so convert the `variance` argument using its square root.

Implement `pull_gauss_np`: return one reward per requested arm in the same order. `pull_np` is provided as an example.

In [ ]:
def pull_np(i_array, means):
    """Sample one Bernoulli reward for each requested arm."""
    return (np.random.rand(len(i_array)) < np.asarray(means)[np.asarray(i_array, dtype=int)]).astype(int)


def pull_gauss_np(i_array, means, variance=1.0):
    """Sample Gaussian rewards with the specified variance, preserving arm order."""
    # <YOUR CODE GOES HERE>
    raise NotImplementedError("Implement pull_gauss_np.")

In [ ]:
from review_checks import check_reward_sampling

check_reward_sampling(pull_np, pull_gauss_np)

Now, let's implement the base class, `BaseBandit`, which we will use later to implement various bandit strategies. In our class, we will store a history of all interactions with the environment. Each entry will be represented by a `HistoryRecord`, describing the numeric reward received from the interaction and the ID of the arm that was pulled. We will also maintain counters for each arm (indicating how many times that particular arm was pulled) as well as the cumulative reward each arm has received.

Implement `add_result` and `choose_arm`. The first `num_arms` interactions must visit arms `0, 1, ..., num_arms - 1` in order; subsequent calls delegate to `next_arm_idx`. Always record an observed reward before choosing again. This initialization prevents division by zero in greedy and UCB policies.

In [ ]:
@dataclass
class HistoryRecord:
    arm_idx: int
    reward: float


class BaseBandit:
    """
    Represents a basic bandit model that selects arms and tracks the history of selections and rewards.

    Attributes
    ----------
    num_arms : int
        The number of arms in the bandit.

    arms_counter : np.ndarray
        Array tracking the number of times each arm has been selected.

    rewards : np.ndarray
        Array tracking the cumulative rewards obtained from each arm.

    label : str
        A label for the bandit instance, defaults to the class name.

    hist : list[HistoryRecord]
        A list of history records detailing each arm selection and its resulting reward.
    """

    def __init__(self, num_arms: int, label: str = None):
        """
        Initialize a new instance of BaseBandit.

        Parameters
        ----------
        num_arms : int
            The number of arms in the bandit.

        label : str, optional
            A label for the bandit instance, by default None (in this case label will be defined as class name).
        """
        self.num_arms = num_arms
        self.arms_counter = np.zeros(self.num_arms, dtype=int)
        self.rewards = np.zeros(self.num_arms)
        self.label = label or self.__class__.__name__
        self.hist: list[HistoryRecord] = []

    def add_result(self, arm_idx: int, reward: float):
        """
        Record the result of pulling an arm.

        Parameters
        ----------
        arm_idx : int
            Index of the arm that was pulled.

        reward : float
            Reward received from pulling the arm.
        """
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement BaseBandit.add_result.")

    def next_arm_idx(self):
        """
        Determine the next arm to pull. To be implemented in subclasses.

        Raises
        ------
        NotImplementedError
            If the method is not overridden in subclasses.
        """
        raise NotImplementedError

    def choose_arm(self):
        """
        Choose the next arm to pull based on bandit's history and strategy.

        Returns
        -------
        int
            Index of the chosen arm.
        """
        # pull each arm at least once
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement BaseBandit.choose_arm.")

    def plot_total(self, y, ax, color: str, label: str, xmax: int):
        """
        Plot the total cumulative values over the bandit's history.

        Parameters
        ----------
        y : array-like
            Values to plot.

        ax : matplotlib Axis
            Axis on which to plot.

        color : str
            Color for the plot.

        label : str
            Label for the plot.

        xmax : int
            Maximum value for the x-axis.
        """
        xs = range(1, len(self.hist) + 1)
        if label is None:
            ax.plot(xs, y, color=color, label=self.label)
        else:
            ax.plot(xs, y, color=color, label=label)
        if xmax > 0:
            ax.set_xlim((1, xmax))

    def plot_history(self, ax, color="C1", label=None, xmax=-1):
        """
        Plot the average rewards over the bandit's history.

        Parameters
        ----------
        ax : matplotlib Axis
            Axis on which to plot.

        color : str, optional
            Color for the plot. Default is "C1".

        label : str, optional
            Label for the plot. By default None.

        xmax : int, optional
            Maximum value for the x-axis. By default -1.
        """
        total = np.cumsum(np.array([x.reward for x in self.hist]))
        avg = np.array([ttl / (i + 1) for i, ttl in enumerate(total)])
        self.plot_total(avg, ax, color, label, xmax)

    def plot_optimal_action(self, ax, opt_arm_idx, color="C1", label=None, xmax=-1):
        """
        Plot the percentage of times the optimal arm was chosen over the bandit's history.

        Parameters
        ----------
        ax : matplotlib Axis
            Axis on which to plot.

        opt_arm_idx : int
            Index of the optimal arm.

        color : str, optional
            Color for the plot. Default is "C1".

        label : str, optional
            Label for the plot. By default None.

        xmax : int, optional
            Maximum value for the x-axis. By default -1.
        """
        total = np.cumsum(
            np.array([x.arm_idx == opt_arm_idx for x in self.hist], dtype=int)
        )
        avg = np.array([ttl / (i + 1) for i, ttl in enumerate(total)])
        self.plot_total(avg, ax, color, label, xmax)

First, we'll implement the `RandomBandit`, which selects the next arm to pull at random.

In [ ]:
class RandomBandit(BaseBandit):
    """
    Represents a bandit strategy where an arm is chosen randomly.

    This class inherits from the BaseBandit class and implements the
    'next_arm_idx' method to randomly select an arm.
    """

    def next_arm_idx(self):
        """
        Select and return a random arm index.

        Returns
        -------
        int
            Index of the randomly chosen arm.
        """
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement RandomBandit.next_arm_idx.")

In [ ]:
from review_checks import check_base_bandit

check_base_bandit(HistoryRecord, BaseBandit, RandomBandit)

### Helper Functions

To compare different bandit strategies, we will implement several helper methods to run the simulation and plot the results (history) of each bandit.

In [ ]:
def run_bandits(bandits, distribution_means, pull=pull_np, n_steps=1000, seed=None, **kwargs):
    """
    Simulate running multiple bandit algorithms over a specified number of steps.

    Parameters
    ----------
    bandits : list
        A list of bandit algorithm instances.

    distribution_means : list
        Expected rewards associated with each arm.

    pull : function, optional
        A function that simulates pulling an arm and returns a reward.
        Default is `pull_np`.

    n_steps : int, optional
        The number of steps to run the simulation. Default is 1000.

    seed : int, optional
        Reset NumPy's random seed before this simulation. None continues the current stream.

    kwargs : dict
        Additional keyword arguments.
    """
    if seed is not None:
        np.random.seed(seed)
    for _ in range(n_steps):
        # For each bandit, determine the next arm to pull
        arms = [b.choose_arm() for b in bandits]

        # Pull the determined arms and get the resulting rewards
        results = pull(arms, distribution_means)

        # Record the results (arm pulled and reward) for each bandit
        for arm_idx, reward, bandit in zip(arms, results, bandits):
            bandit.add_result(arm_idx, reward)

In [ ]:
def plot_simulation_results_over_single_type(
    bandits: list[BaseBandit],
    opt_arm_idx: int,
    n_steps: int,
    loc="upper right",
    figsize=(14, 5),
    **kwargs,
):
    """
    Plot historical data for multiple bandits of the same type (e.g. RandomBandit) for comparison.

    Parameters
    ----------
    bandits : list[BaseBandit]
        A list of bandit instances to be plotted.

    opt_arm_idx : int
        Index of the optimal arm.

    n_steps : int
        Number of steps for the x-axis.

    loc : str, optional
        Legend location. Default is "upper right".

    figsize : tuple, optional
        Figure size. Default is (14, 5).
    """
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    for i, bandit in enumerate(bandits):
        label = f"{bandit.label}, run {i + 1}"
        bandit.plot_history(ax[0], f"C{i}", label=label, xmax=n_steps)
        bandit.plot_optimal_action(ax[1], opt_arm_idx, f"C{i}", label=label, xmax=n_steps)

    ax[0].set_title("Average Reward")
    ax[1].set_title("Proportion of Optimal Choices")

    ax[0].set_ylabel("Cumulative average reward")
    ax[1].set_ylabel("Cumulative optimal-choice proportion")
    ax[1].set_ylim(0, 1)
    for a in ax:
        a.set_xlabel("Step")
        a.set_xlim((0, n_steps))
        a.legend(loc=loc)
    fig.tight_layout()
    return fig, ax


def run_aggregated_simulation_over_single_type(
    distribution_means: list[float], num_steps: int, bandits: list[BaseBandit], **kwargs
):
    """
    Simulate running multiple bandit algorithms and plot their performance.

    Parameters
    ----------
    distribution_means : list[float]
        Expected rewards associated with each arm.

    num_steps : int
        Number of steps for the simulation.

    bandits : list[BaseBandit]
        A list of bandit instances to be simulated.
    """
    distribution_means = np.array(distribution_means)
    opt_arm_idx = np.argmax(distribution_means)
    run_bandits(bandits, distribution_means, n_steps=num_steps, **kwargs)
    fig, ax = plot_simulation_results_over_single_type(
        bandits, opt_arm_idx, num_steps, **kwargs
    )
    return fig, ax


def plot_simulation_results_over_multiple_types(
    bandits: list[list[BaseBandit]],
    opt_arm_idx,
    n_steps: int,
    loc="upper right",
    figsize=(14, 5),
    **kwargs,
):
    """
    Plot averaged data over multiple runs of bandit instances.

    Parameters
    ----------
    bandits : list[list[BaseBandit]]
        List of lists of bandit instances.

    opt_arm_idx : int
        Index of the optimal arm.

    n_steps : int
        Number of steps for the x-axis.

    loc : str, optional
        Legend location. Default is "upper right".

    figsize : tuple, optional
        Figure size. Default is (14, 5).
    """
    fig, axs = plt.subplots(1, 2, figsize=figsize)
    axs = axs.ravel()

    for i, bs in enumerate(bandits):
        total = np.cumsum(np.array([[x.reward for x in b.hist] for b in bs]), axis=1)
        total_arm = np.cumsum(
            np.array(
                [[x.arm_idx == opt_arm_idx for x in b.hist] for b in bs], dtype=int
            ),
            axis=1,
        )
        avg = np.true_divide(total, np.arange(1, n_steps + 1)).mean(axis=0)
        avg_arm = np.true_divide(total_arm, np.arange(1, n_steps + 1)).mean(axis=0)

        bs[0].plot_total(avg, axs[0], f"C{i}", bs[0].label, xmax=n_steps)
        bs[0].plot_total(avg_arm, axs[1], f"C{i}", bs[0].label, xmax=n_steps)

    total_bandits = len(bandits[0] if bandits else [])
    axs[0].set_title(f"Average Reward (aggregated over {total_bandits} instances)")
    axs[1].set_title(
        f"Proportion of Optimal Choices (aggregated over {total_bandits} instances)"
    )
    axs[0].set_ylabel("Cumulative average reward")
    axs[1].set_ylabel("Cumulative optimal-choice proportion")
    axs[1].set_ylim(0, 1)
    for ax in axs:
        ax.set_xlabel("Step")
        ax.set_xlim((0, n_steps))
        ax.legend(loc=loc)
    fig.tight_layout()
    return fig, axs


def run_aggregated_simulation_over_multiple_types(
    distribution_means: list[float],
    num_steps: int,
    bandits: list[list[BaseBandit]],
    **kwargs,
):
    """
    Simulate and plot average performance over multiple runs of bandit instances.

    Parameters
    ----------
    distribution_means : list[float]
        Expected rewards associated with each arm.

    num_steps : int
        Number of steps for the simulation.

    bandits : list[list[BaseBandit]]
        List of lists of bandit instances.
    """
    distribution_means = np.array(distribution_means)
    opt_arm_idx = np.argmax(distribution_means)
    for bs in bandits:
        run_bandits(bs, distribution_means, n_steps=num_steps, **kwargs)
    fig, ax = plot_simulation_results_over_multiple_types(
        bandits, opt_arm_idx, num_steps, **kwargs
    )
    return fig, ax

Let's develop a comprehensive function named `conduct_experiment` that integrates all of the previously defined functionalities.

This function will accept the following inputs:

- `BanditSettings`: A list containing descriptions of the bandit classes for which the simulation will be run
- `num_bandits_per_type`: The number of instances for each type of bandit to be created. This parameter is used to average performance across multiple instances and yield more robust results
- `num_steps`: The total number of steps for which the simulation will be run
- `distribution_means`: An array of means from which the rewards will be sampled, corresponding *to each arm*
- `num_bandits_for_sanity_check`: A subset of bandits for each BanditSetting that will be plotted to perform a sanity check, illustrating how different instances of similar bandits can yield varied performances due to the strategies' randomness.
- `is_gauss`: A boolean flag indicating whether the underlying reward distribution should be Normal (Gaussian) or Bernoullian.
- `seed`: An integer random seed, reset at the start of each experiment (default `SEED`).
- `kwargs`: A dictionary of additional parameters that will be passed to visualization functions.

`conduct_experiment` returns the groups of simulated bandits so that you can inspect their histories. By default it uses Gaussian rewards; set `is_gauss=False` for Bernoulli experiments. Beta-Bernoulli Thompson sampling must only be used with Bernoulli rewards.

In [ ]:
@dataclass
class BanditSettings:
    """Used to describe a single bandit instance."""

    bandit_cls: type[BaseBandit]
    params: dict = field(default_factory=dict)


def create_single_type_bandits(
    setting: BanditSettings,
    num_bandits_per_type: int,
    n_arms: int,
) -> list[BaseBandit]:
    return [
        setting.bandit_cls(n_arms, **setting.params)
        for _ in range(num_bandits_per_type)
    ]


def conduct_experiment(
    bandit_settings: list[BanditSettings],
    num_bandits_per_type: int,
    num_steps: int,
    distribution_means: list[float],
    num_bandits_for_sanity_check: int = 3,
    is_gauss: bool = True,
    seed: int = SEED,
    **kwargs,
):
    """Run reproducible experiments and return the aggregated groups of bandits."""
    np.random.seed(seed)
    pull = pull_gauss_np if is_gauss else pull_np
    n_arms = len(distribution_means)
    if num_bandits_for_sanity_check:
        for setting in bandit_settings:
            # for each bandit type, plot `num_bandits_for_sanity_check` distinct plots of the same type
            bandits = create_single_type_bandits(
                setting, num_bandits_for_sanity_check, n_arms
            )
            run_aggregated_simulation_over_single_type(
                distribution_means, num_steps, bandits, pull=pull, **kwargs
            )

    # aggregate results for each type
    bandits = [
        create_single_type_bandits(setting, num_bandits_per_type, n_arms)
        for setting in bandit_settings
    ]
    run_aggregated_simulation_over_multiple_types(
        distribution_means, num_steps, bandits, pull=pull, **kwargs
    )
    return bandits

In [ ]:
from review_checks import check_experiments

check_experiments(
    RandomBandit, BanditSettings, run_bandits,
    plot_simulation_results_over_single_type, conduct_experiment,
)

### Run Random Simulations

Run multiple `RandomBandit` instances. The individual-run plots illustrate variation within an experiment; the comparison plot averages over all instances. Change `seed` explicitly to obtain a new experiment.

In [ ]:
conduct_experiment(
    bandit_settings=[BanditSettings(RandomBandit)],
    num_bandits_per_type=200,
    num_steps=5000,
    distribution_means=[0.3, 0.55, 0.65],
);

## Greedy Bandit

Implement `GreedyBandit.next_arm_idx`: select the arm with the highest empirical mean reward. `BaseBandit.choose_arm` guarantees that every arm has already been pulled once. Use `np.argmax` to break ties by the first maximum.

In [ ]:
class GreedyBandit(BaseBandit):
    """
    Represents a multi-armed bandit strategy that selects the arm with the
    highest estimated reward value.

    The GreedyBandit strategy chooses the arm based on the past rewards it
    has received. Specifically, it divides the total rewards for each arm
    by the number of times that arm has been chosen, and then selects the
    arm with the highest average reward.
    """

    def next_arm_idx(self):
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement GreedyBandit.next_arm_idx.")

In [ ]:
from review_checks import check_greedy_bandit

check_greedy_bandit(GreedyBandit)

In [ ]:
conduct_experiment(
    bandit_settings=[BanditSettings(GreedyBandit)],
    num_bandits_per_type=200,
    num_steps=5000,
    distribution_means=[0.3, 0.55, 0.65],
    num_bandits_for_sanity_check=5,
    loc="center right",
);

In [ ]:
conduct_experiment(
    bandit_settings=[BanditSettings(GreedyBandit), BanditSettings(RandomBandit)],
    num_bandits_per_type=200,
    num_steps=5000,
    distribution_means=[0.3, 0.55, 0.65],
    num_bandits_for_sanity_check=5,
    loc="center right",
);

## $\varepsilon$-greedy

Implement `EpsGreedyBandit.next_arm_idx`: with probability $\varepsilon$, select uniformly among **all** arms (including the current greedy arm); otherwise, choose the arm with the highest empirical mean reward. At $\varepsilon=0$ this is greedy; at $\varepsilon=1$ this is uniform random selection after initialization.

In [ ]:
class EpsGreedyBandit(BaseBandit):
    """
    Represents the ε-greedy bandit strategy. This strategy primarily exploits the
    best-performing arm but occasionally explores other arms based on a probability ε.

    Attributes
    ----------
    eps : float
        The exploration probability. A value between 0 (no exploration, purely greedy)
        and 1 (pure exploration, random choice of arms).
    """

    def __init__(self, num_arms: int, label: str = None, eps: float = 1.0):
        """
        Initialize the EpsGreedyBandit with a given number of arms and exploration probability ε.

        Parameters
        ----------
        num_arms : int
            The total number of arms in the bandit.
        label : str, optional
            A label for the bandit strategy.
        eps : float, optional
            The exploration probability. Default is 1.0 (pure exploration).
        """
        super().__init__(
            num_arms, label or f"$\\epsilon$-greedy strategy, $\\epsilon={eps}$"
        )
        if eps > 1.0 or eps < 0.0:
            raise ValueError("Eps value must be between 0 and 1")
        self.eps = eps

    def next_arm_idx(self):
        """
        Choose the next arm to pull based on the ε-greedy strategy.

        Returns
        -------
        int
            The index of the arm to pull next.
        """
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement EpsGreedyBandit.next_arm_idx.")

In [ ]:
from review_checks import check_epsilon_greedy

check_epsilon_greedy(EpsGreedyBandit)

In [ ]:
conduct_experiment(
    bandit_settings=[
        BanditSettings(EpsGreedyBandit, params={"eps": eps})
        for eps in (0.2, 0.1, 0.05, 0.01)
    ],
    num_bandits_per_type=100,
    num_steps=15_000,
    distribution_means=[0.3, 0.55, 0.65],
    num_bandits_for_sanity_check=5,
    loc="center right",
);

In [ ]:
conduct_experiment(
    bandit_settings=[
        BanditSettings(EpsGreedyBandit, params={"eps": eps})
        for eps in (0.2, 0.1, 0.05, 0.01)
    ],
    num_bandits_per_type=100,
    num_steps=15_000,
    distribution_means=[0.5, 0.75, 1.0, 1.15],
    num_bandits_for_sanity_check=5,
    loc="center right",
);

Let's compare different `EpsGreedyBandit`, `GreedyBandit` and `RandomBandit` with each other.

In [ ]:
settings = []
for eps in (0.2, 0.1, 0.05, 0.01):
    settings.append(BanditSettings(EpsGreedyBandit, params={"eps": eps}))
settings.append(BanditSettings(GreedyBandit))
settings.append(BanditSettings(RandomBandit))

conduct_experiment(
    bandit_settings=settings,
    num_bandits_per_type=100,
    num_steps=5000,
    distribution_means=[0.3, 0.5, 0.6, 0.55],
    num_bandits_for_sanity_check=0,
    loc="lower right",
);

## UCB1

UCB selects an arm using both its empirical mean and an uncertainty bonus. Less frequently sampled arms receive a larger bonus:

$$
A_t = \arg\max_a \left[\widehat{\mu}_a(t) + c\sqrt{\frac{\ln t}{N_a(t)}}\right],
$$

where $t$ is the number of observations already in `hist`, $N_a(t)$ is the arm's pull count, and $\widehat{\mu}_a(t)$ is its empirical mean. Each arm is pulled once before this rule is used. Implement `UCB1Bandit.next_arm_idx` using these quantities.

For rewards in $[0, 1]$, $c=\sqrt{2}$ gives the UCB1 rule studied by [Auer, Cesa-Bianchi and Fischer (2002), *Finite-time Analysis of the Multiarmed Bandit Problem* (open author PDF)](https://homes.di.unimi.it/~cesabian/Pubblicazioni/ml-02.pdf). Other values of $c$ tune exploration. We also compare the same rule empirically on Gaussian rewards; the paper's bounded-reward assumptions do not apply to those unbounded rewards.

In [ ]:
class UCB1Bandit(BaseBandit):
    """
    Implements the Upper Confidence Bound (UCB1) multi-armed bandit strategy.

    The UCB1 algorithm is designed to balance exploration and exploitation by
    selecting arms that have the highest upper confidence bounds. Specifically,
    it selects the arm that maximizes the sum of its empirical mean reward and
    an uncertainty term that decreases as the arm is pulled more often.

    Attributes
    ----------
    c : float
        The exploration parameter which determines the degree of exploration
        versus exploitation. A higher value results in more exploration.
    """

    def __init__(self, num_arms: int, label: str = None, c: float = np.sqrt(2)):
        super().__init__(num_arms, label or f"UCB1 ({c:.4f})")
        self.c = c

    def next_arm_idx(self):
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement UCB1Bandit.next_arm_idx.")

In [ ]:
from review_checks import check_ucb1

check_ucb1(UCB1Bandit)

In [ ]:
settings = [
    BanditSettings(GreedyBandit),
    BanditSettings(EpsGreedyBandit, params={"eps": 0.01}),
    BanditSettings(EpsGreedyBandit, params={"eps": 0.05}),
    BanditSettings(EpsGreedyBandit, params={"eps": 0.1}),
    BanditSettings(UCB1Bandit, params={"c": 1.0}),
    BanditSettings(UCB1Bandit, params={"c": np.sqrt(2)}),
]

conduct_experiment(
    bandit_settings=settings,
    num_bandits_per_type=100,
    num_steps=5000,
    distribution_means=[0.3, 0.4, 0.5, 0.6],
    num_bandits_for_sanity_check=1,
    loc="center right",
);

## Thompson sampling

**Thompson Sampling**, also known as posterior sampling, is a probability-based algorithm used for solving the multi-armed bandit problem.

The fundamental idea behind Thompson Sampling is to maintain a probability distribution over the expected reward of each arm and, at each step, pull the arm that has the highest potential reward according to a sample from these distributions.

Here's a step-by-step breakdown:

- **Initialization**: For each arm, we start with a prior distribution over its expected reward. This is typically chosen to be a Beta distribution, **due to its conjugate property with the Bernoulli distribution (which describes the reward process of each arm).**
- **Sampling**: For each arm, draw a sample from its current distribution. This gives us a potential reward for each arm.
- **Selection**: Pull the arm that has the highest sampled reward.
- **Update**: After observing the reward of the pulled arm, update its distribution. The distribution moves closer to the true reward of the arm with each pull, refining our knowledge over time.

In this lab, implement `ThompsonBandit.next_arm_idx` and `ThompsonBandit.add_result` for **Bernoulli rewards only**. Start with independent $\operatorname{Beta}(1, 1)$ priors. A success increments $\alpha$ and a failure increments $\beta$ for the selected arm; also update the base history and counters. Reject non-binary rewards before changing any state. Draw one posterior sample per arm with `np.random.beta(self.alpha, self.beta)` and return the index of the largest sample.

As for the other strategies, `BaseBandit.choose_arm` first pulls each arm once. The pseudocode below includes this shared initialization; those rewards must update the posteriors too.

See [Russo et al., *A Tutorial on Thompson Sampling*](https://arxiv.org/abs/1707.02038) and [Agrawal and Goyal, *Analysis of Thompson Sampling for the Multi-armed Bandit Problem*](https://arxiv.org/abs/1111.1797), both available in open access.

```
Initialize for each arm a (indices 0, ..., num_arms - 1):
    trials[a] = 0
    rewards[a] = 0
    alpha[a] = 1
    beta[a] = 1

For step = 1 to num_steps:
    If step <= num_arms:
        chosen_arm = step - 1        // Pull each arm once
    Else:
        For each arm a:
            theta[a] = Sample from Beta(alpha[a], beta[a])
        chosen_arm = argmax_a(theta[a])

    r = Pull(chosen_arm)             // Binary reward: 0 or 1
    Append (chosen_arm, r) to history
    trials[chosen_arm] += 1
    rewards[chosen_arm] += r
    alpha[chosen_arm] += r
    beta[chosen_arm] += 1 - r
```

In [ ]:
class ThompsonBandit(BaseBandit):
    """
    Represents a bandit using the Thompson Sampling strategy.
    Uses Beta priors and binary Bernoulli rewards.
    """

    def __init__(self, num_arms: int, label: str = None):
        super().__init__(num_arms, label)
        # Initialize alpha and beta for each arm's Beta distribution (start with 1,1 which is a uniform distribution)
        self.alpha = np.ones(self.num_arms)
        self.beta = np.ones(self.num_arms)

    def next_arm_idx(self):
        """
        Determine the next arm to pull using Thompson Sampling.

        Returns
        -------
        int
            Index of the chosen arm.
        """
        # Sample from the current beta distributions of each arm
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement ThompsonBandit.next_arm_idx.")

    def add_result(self, arm_idx: int, reward: float):
        """
        Record the result of pulling an arm and update Beta distribution parameters.

        Parameters
        ----------
        arm_idx : int
            Index of the arm that was pulled.

        reward : float
            Reward received from pulling the arm.
        """
        # <YOUR CODE GOES HERE>
        raise NotImplementedError("Implement ThompsonBandit.add_result.")

    def plot_distributions(self, ax, step):
        """
        Plot the probability distributions for each arm based on the alpha and beta parameters at timestamp `step`.
        Use `self.hist` to obtain the state of `self.alpha` and `self.beta` at step `step`.

        Parameters:
        - ax: The axis on which to plot the distributions.
        - step: The step in the simulation.
        """
        if not 0 <= step <= len(self.hist):
            raise ValueError("Step must be between 0 and the history length")
        xs = np.linspace(0.001, 0.999, 100)
        alpha, beta = np.ones(self.num_arms), np.ones(self.num_arms)
        rewards, arms_counter = np.zeros(self.num_arms), np.zeros(
            self.num_arms, dtype=int
        )

        for step_id in range(step):
            history_record = self.hist[step_id]
            rewards[history_record.arm_idx] += history_record.reward
            arms_counter[history_record.arm_idx] += 1

            if history_record.reward > 0:
                alpha[history_record.arm_idx] += history_record.reward
            else:
                beta[history_record.arm_idx] += 1 - history_record.reward

        for arm_idx in range(self.num_arms):
            prior = st.beta(a=alpha[arm_idx], b=beta[arm_idx])
            y = prior.pdf(xs)
            ax.plot(
                xs,
                y,
                color=f"C{arm_idx}",
                label=f"Arm {arm_idx + 1}: mean={prior.mean():.3f}, successes/pulls={int(rewards[arm_idx])}/{int(arms_counter[arm_idx])}",
            )
            ax.fill_between(xs, y, 0, color=f"C{arm_idx}", alpha=0.2)

        ax.set_xlabel("Bernoulli success probability")
        ax.set_ylabel("Posterior density")
        ax.set_xlim([0, 1])
        ax.set_ylim(bottom=0)
        ax.legend(loc="upper left")
        ax.set_title(f"Distribution on Step {step}")

In [ ]:
from review_checks import check_thompson_sampling

check_thompson_sampling(ThompsonBandit)

Let's see how the estimation of the current params changes as we observe more results from the bandit.

In [ ]:
fig, axs = plt.subplots(5, 2, figsize=(12, 15))
axs = axs.flat

thompson_bandit = ThompsonBandit(3)
run_bandits(
    [thompson_bandit],
    np.array([0.3, 0.55, 0.65]),
    pull=pull_np,
    n_steps=5000,
    seed=SEED,
)

for step in (1, 2, 5, 10, 25, 50, 100, 200, 500, 5000):
    thompson_bandit.plot_distributions(next(axs), step)

fig.tight_layout()
plt.show()

Compare the posterior means with the true success probabilities `[0.3, 0.55, 0.65]`. Arms pulled less often can retain broad posteriors, even after many total steps; a single trajectory does not guarantee accurate estimates for every arm.

Now compare UCB1 and Thompson sampling on **Bernoulli** rewards, averaging over 100 independent instances per strategy. Use the questions at the end to interpret the results and their dependence on the seed.

In [ ]:
settings = [
    BanditSettings(ThompsonBandit),
    BanditSettings(UCB1Bandit, params={"c": 1.0}),
    BanditSettings(UCB1Bandit, params={"c": np.sqrt(2)}),
]

conduct_experiment(
    bandit_settings=settings,
    num_bandits_per_type=100,
    num_steps=5000,
    distribution_means=[0.3, 0.55, 0.65],
    num_bandits_for_sanity_check=3,
    is_gauss=False,
    loc="center right",
);

## Questions

1. Why can `GreedyBandit` get stuck choosing a suboptimal arm? Use the individual-run plots to explain how early rewards affect its later choices.
2. Compare $\varepsilon=0.1$ and $\varepsilon=0.01$ in the Gaussian experiments. How does the exploration probability affect average reward and optimal-arm choices over short and long horizons?
3. How do UCB1 and Thompson sampling differ in how they choose an arm to explore? Relate this difference to their results in the final Bernoulli experiment.

## Your answers

Your answer goes here.